# Lab Assignment: Landmark Recognition Using SIFT

This notebook builds a small tourism information system. You give it a photo, and it tells you which famous landmark is in it (or says "unknown"). It uses Python and OpenCV only.

**How the pipeline works**

1. Build a database of reference photos, one folder per landmark.
2. Run SIFT on every reference image and store the keypoints and descriptors.
3. Run SIFT on the query image.
4. Match query descriptors to each landmark with a Brute-Force matcher (Euclidean distance), k-NN with k=2, and Lowe's ratio test.
5. Check the best candidates with a RANSAC homography and draw the landmark outline on the query.
6. Decide: recognized or rejected, using thresholds (not just the biggest match count).
7. Test under different conditions and record the results.

**Folder layout this notebook creates**

```
landmark_project/
    landmark_db/          reference images, one folder per landmark
        Eiffel Tower/
        Taj Mahal/
        ...
    landmark_tests/       held-out photos used only for testing
        Eiffel Tower/
        ...
        _unknown/         landmarks that are NOT in the database
    outputs/              saved figures and the results table
    sift_database.pkl     stored descriptors with labels
```

If you want to use your own photos, just put them in these folders before running Task 2. The download cell skips any landmark that already has enough images.

In [ ]:
import os, time, json, pickle, urllib.request, urllib.parse
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("OpenCV version:", cv2.__version__)

# ---------- folders ----------
BASE     = Path("landmark_project")
DB_DIR   = BASE / "landmark_db"
TEST_DIR = BASE / "landmark_tests"
OUT_DIR  = BASE / "outputs"
for d in (DB_DIR, TEST_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---------- landmarks (folder name -> search words for Wikimedia Commons) ----------
LANDMARKS = {
    "Eiffel Tower":        "Eiffel Tower Paris",
    "Taj Mahal":           "Taj Mahal Agra",
    "Great Wall of China": "Great Wall of China Badaling",
    "Statue of Liberty":   "Statue of Liberty New York",
    "Colosseum":           "Colosseum Rome",
}
# landmarks that are NOT in the database (used to test rejection)
UNKNOWN_LANDMARKS = {
    "Pyramids of Giza":   "Great Pyramid of Giza",
    "Sydney Opera House": "Sydney Opera House",
}

N_DB   = 5     # reference images per landmark
N_TEST = 2     # held-out test images per landmark (never added to the database)

# ---------- SIFT and matching settings ----------
MAX_SIDE      = 900    # resize big photos so the longest side is at most this
N_FEATURES    = 3000   # cap on SIFT keypoints per image (keeps it fast)
RATIO         = 0.75   # Lowe's ratio threshold
RANSAC_THRESH = 5.0    # reprojection error in pixels
TOP_K         = 3      # how many top candidates get geometric verification

# ---------- decision thresholds ----------
MIN_GOOD         = 12    # at least this many reliable matches
MIN_INLIERS      = 10    # at least this many RANSAC inliers
MIN_INLIER_RATIO = 0.25  # inliers / reliable matches
MARGIN           = 1.5   # best candidate needs 1.5x the inliers of the runner-up

## Task 1: Landmark image database

The next cell downloads photos from Wikimedia Commons: `N_DB` reference images and `N_TEST` held-out test images for each landmark, plus a couple of images of landmarks that are *not* in the database. It needs internet access.

Download problems are normal (rate limits, a blocked network). If it fails, copy your own photos into the folders shown above and run the notebook again.

In [ ]:
UA = {"User-Agent": "LandmarkSIFTLab/1.0 (university lab assignment)"}
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}

def list_images(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in IMG_EXT)

def commons_search(query, limit=40):
    """Return image URLs (JPEG, reasonably large) from Wikimedia Commons for a search phrase."""
    params = {
        "action": "query", "generator": "search",
        "gsrsearch": f"{query} filetype:bitmap", "gsrnamespace": 6, "gsrlimit": limit,
        "prop": "imageinfo", "iiprop": "url|size|mime", "iiurlwidth": 1000, "format": "json",
    }
    url = "https://commons.wikimedia.org/w/api.php?" + urllib.parse.urlencode(params)
    with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=30) as r:
        data = json.load(r)
    pages = data.get("query", {}).get("pages", {})
    urls = []
    for p in sorted(pages.values(), key=lambda p: p.get("index", 0)):
        info = (p.get("imageinfo") or [{}])[0]
        if info.get("mime") != "image/jpeg":
            continue
        if min(info.get("width", 0), info.get("height", 0)) < 500:
            continue
        urls.append(info.get("thumburl") or info.get("url"))
    return urls

def download_image(url, dest):
    for attempt in range(3):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=60) as r:
                dest.write_bytes(r.read())
            if cv2.imread(str(dest)) is not None:
                return True
            dest.unlink(missing_ok=True)
        except Exception:
            time.sleep(2 * (attempt + 1))
    return False

def fetch_images(search, slug, targets):
    """targets = list of (folder, how_many). Fills each folder in order from the search results."""
    need = [(Path(f), n) for f, n in targets if len(list_images(f)) < n]
    if not need:
        return
    try:
        urls = commons_search(search)
    except Exception as e:
        print(f"  could not search for '{search}': {e}")
        return
    it = iter(urls)
    for folder, n in need:
        folder.mkdir(parents=True, exist_ok=True)
        have = len(list_images(folder))
        while have < n:
            url = next(it, None)
            if url is None:
                break
            if download_image(url, folder / f"{slug}_{have+1:02d}.jpg"):
                have += 1
            time.sleep(0.5)

for name, search in LANDMARKS.items():
    slug = name.lower().replace(" ", "_")
    fetch_images(search, slug, [(DB_DIR / name, N_DB), (TEST_DIR / name, N_TEST)])
for name, search in UNKNOWN_LANDMARKS.items():
    slug = name.lower().replace(" ", "_")
    fetch_images(search, slug, [(TEST_DIR / "_unknown" / slug, 1)])

def unknown_images():
    return sorted(p for p in (TEST_DIR / "_unknown").rglob("*") if p.suffix.lower() in IMG_EXT)

# ---------- check what we have ----------
problems = [n for n in LANDMARKS if len(list_images(DB_DIR / n)) < 2]
for name in LANDMARKS:
    print(f"{name:22s} database: {len(list_images(DB_DIR / name))}   tests: {len(list_images(TEST_DIR / name))}")
print("unknown test images:", len(unknown_images()))
if problems:
    raise SystemExit(f"Not enough images for: {problems}. Add photos to {DB_DIR}/<landmark>/ and re-run.")

**Look at your database.** Wikimedia search results are not perfect, so quickly check that each row really shows that landmark. Delete any wrong image from its folder and re-run the cell above if needed. This figure is also your Task 1 screenshot.

In [ ]:
def load_img(path, max_side=MAX_SIDE):
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(path)
    h, w = img.shape[:2]
    s = max_side / max(h, w)
    if s < 1:
        img = cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
    return img

def rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

cols = max(len(list_images(DB_DIR / n)) for n in LANDMARKS)
fig, axes = plt.subplots(len(LANDMARKS), cols, figsize=(3 * cols, 2.6 * len(LANDMARKS)), squeeze=False)
for r, name in enumerate(LANDMARKS):
    paths = list_images(DB_DIR / name)
    for c in range(cols):
        ax = axes[r][c]; ax.axis("off")
        if c < len(paths):
            ax.imshow(rgb(load_img(paths[c], 400)))
        if c == 0:
            ax.set_title(name, loc="left", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "task1_database.png", dpi=120)
plt.show()

## Task 2: SIFT feature extraction

For every reference image: load it, convert to grayscale, detect keypoints, compute 128-dimensional descriptors, and store them with the landmark label.

SIFT finds blob-like points that look the same even if the image is scaled or rotated, and describes the area around each point with a 128-number histogram of gradient directions.

In [ ]:
sift = cv2.SIFT_create(nfeatures=N_FEATURES)

def extract_sift(img_bgr):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)   # step 1: grayscale
    kp, des = sift.detectAndCompute(gray, None)         # steps 2 and 3: keypoints and descriptors
    return gray, kp, des

# --- helpers to save keypoints (cv2.KeyPoint objects cannot be pickled directly) ---
def kp_to_tuple(k):
    return (k.pt, k.size, k.angle, k.response, k.octave, k.class_id)

def tuple_to_kp(t):
    return cv2.KeyPoint(x=t[0][0], y=t[0][1], size=t[1], angle=t[2], response=t[3], octave=t[4], class_id=t[5])

def build_database():
    db = {}
    for name in LANDMARKS:
        entries = []
        for path in list_images(DB_DIR / name):
            img = load_img(path)
            gray, kp, des = extract_sift(img)
            if des is None or len(kp) < 2:
                print("  skipped (no features):", path.name)
                continue
            entries.append({"file": path.name, "path": str(path), "shape": img.shape[:2],
                            "kp": [kp_to_tuple(k) for k in kp], "des": des})
        db[name] = entries
    return db

t0 = time.perf_counter()
raw_db = build_database()
print(f"Extracted features in {time.perf_counter() - t0:.1f} s")

# store descriptors with their landmark labels
with open(BASE / "sift_database.pkl", "wb") as f:
    pickle.dump(raw_db, f)

# load it back, the way a real application would, and rebuild keypoint objects
with open(BASE / "sift_database.pkl", "rb") as f:
    database = pickle.load(f)
for name, entries in database.items():
    for e in entries:
        e["kp"] = [tuple_to_kp(t) for t in e["kp"]]
        e["pts"] = np.float32([k.pt for k in e["kp"]])

rows = [{"Landmark": n, "Image": e["file"], "Keypoints": len(e["kp"]), "Descriptor shape": e["des"].shape}
        for n, es in database.items() for e in es]
pd.DataFrame(rows)

In [ ]:
# Display one reference image per landmark: grayscale and with SIFT keypoints
fig, axes = plt.subplots(2, len(LANDMARKS), figsize=(3.6 * len(LANDMARKS), 6.5))
for c, name in enumerate(LANDMARKS):
    e = database[name][0]
    img = load_img(e["path"])
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    vis = cv2.drawKeypoints(img, e["kp"], None, color=(0, 255, 0), flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    axes[0][c].imshow(gray, cmap="gray"); axes[0][c].set_title(name + "\n(grayscale)", fontsize=10)
    axes[1][c].imshow(rgb(vis));          axes[1][c].set_title(f"{len(e['kp'])} keypoints", fontsize=10)
    axes[0][c].axis("off"); axes[1][c].axis("off")
plt.tight_layout()
plt.savefig(OUT_DIR / "task2_reference_keypoints.png", dpi=120)
plt.show()

# One descriptor looks like this: 128 numbers
e = database[list(LANDMARKS)[0]][0]
print("First descriptor of", e["file"], "has", e["des"].shape[1], "numbers:")
print(np.round(e["des"][0][:16], 1), "...")

## Task 3: Query image processing

`process_query` takes the path of an unknown image (or an image array), converts it to grayscale, detects SIFT keypoints, extracts descriptors, and shows the keypoints.

In [ ]:
def process_query(source, show=True, title="Query image"):
    img = load_img(source) if isinstance(source, (str, Path)) else source
    gray, kp, des = extract_sift(img)
    if show:
        vis = cv2.drawKeypoints(img, kp, None, color=(0, 255, 0), flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
        fig, ax = plt.subplots(1, 2, figsize=(12, 5))
        ax[0].imshow(rgb(img)); ax[0].set_title(title); ax[0].axis("off")
        ax[1].imshow(rgb(vis)); ax[1].set_title(f"Detected SIFT keypoints: {len(kp)}"); ax[1].axis("off")
        plt.tight_layout(); plt.show()
    return img, kp, des

# Demo with the first held-out test photo of the first landmark
demo_name = list(LANDMARKS)[0]
demo_path = list_images(TEST_DIR / demo_name)[0]
q_img, q_kp, q_des = process_query(demo_path, title=f"Query (true label: {demo_name})")
print("Descriptor matrix:", q_des.shape)

## Task 4: Feature matching and recognition

For each reference image we use:

- **Brute-Force matcher** with `NORM_L2` (Euclidean distance)
- **k-NN with k=2**: for each query descriptor we get its two closest reference descriptors
- **Lowe's ratio test**: keep a match only if the best distance is clearly smaller than the second best (`best < 0.75 * second`). This removes ambiguous matches.

Each landmark gets a score: the highest number of reliable matches over its reference images. We rank landmarks by this score.

**Important:** the final decision does *not* use the match count alone. A wrong landmark can collect many scattered matches by chance (repeating textures like windows, arches or bricks). So in Task 5 the top candidates must also pass a geometric check (RANSAC), and the winner must clearly beat the runner-up. Thresholds are set at the top of the notebook.

In [ ]:
bf = cv2.BFMatcher(cv2.NORM_L2)

def ratio_test_matches(des_q, des_r, ratio=RATIO):
    """kNN (k=2) + Lowe's ratio test. Also keeps only one match per reference keypoint."""
    if des_q is None or des_r is None or len(des_q) < 2 or len(des_r) < 2:
        return []
    knn = bf.knnMatch(des_q, des_r, k=2)
    good = [p[0] for p in knn if len(p) == 2 and p[0].distance < ratio * p[1].distance]
    best_per_ref = {}                      # several query points can hit the same reference point
    for m in good:
        if m.trainIdx not in best_per_ref or m.distance < best_per_ref[m.trainIdx].distance:
            best_per_ref[m.trainIdx] = m
    return list(best_per_ref.values())

def score_landmarks(des_q):
    scores = {}
    for name, entries in database.items():
        best = {"good": -1}
        for e in entries:
            m = ratio_test_matches(des_q, e["des"])
            if len(m) > best["good"]:
                best = {"good": len(m), "entry": e, "matches": m}
        scores[name] = best
    return scores

scores = score_landmarks(q_des)
ranking = sorted(scores, key=lambda n: scores[n]["good"], reverse=True)

print("Ranking by reliable matches:")
for i, n in enumerate(ranking, 1):
    print(f"  {i}. {n:22s} {scores[n]['good']:4d}  (best image: {scores[n]['entry']['file']})")

plt.figure(figsize=(7, 3.5))
plt.barh(ranking[::-1], [scores[n]["good"] for n in ranking[::-1]], color="#4C78A8")
plt.axvline(MIN_GOOD, color="red", ls="--", label=f"minimum = {MIN_GOOD}")
plt.xlabel("Reliable matches"); plt.title("Landmark candidates"); plt.legend()
plt.tight_layout(); plt.savefig(OUT_DIR / "task4_ranking.png", dpi=120); plt.show()
print(f"\nMost likely landmark so far: {ranking[0]} with {scores[ranking[0]]['good']} reliable matches (not final yet, see Task 5)")

## Task 5: Landmark localization (geometric verification)

For a candidate landmark we take the matched points, estimate a **homography** with RANSAC, and project the four corners of the reference image into the query image. If the transformation looks sane, we draw the polygon. If not, we print a message and draw nothing.

A homography is rejected when:
- RANSAC cannot find one, or
- the polygon is not convex, is far too small or too big, or has a tiny edge.

Final decision rules (all must be true to say "RECOGNIZED"):
1. reliable matches >= `MIN_GOOD`
2. RANSAC inliers >= `MIN_INLIERS`
3. inliers / reliable matches >= `MIN_INLIER_RATIO`
4. the projected polygon is valid
5. inliers >= `MARGIN` times the inliers of the runner-up landmark

In [ ]:
def check_polygon(poly, q_shape):
    h, w = q_shape[:2]
    if not np.all(np.isfinite(poly)):
        return False, "projected corners are not finite"
    cnt = poly.astype(np.float32).reshape(-1, 1, 2)
    if not cv2.isContourConvex(cnt):
        return False, "projected polygon is not convex (twisted)"
    area = cv2.contourArea(cnt)
    if area < 0.01 * w * h:
        return False, "projected polygon is too small"
    if area > 10 * w * h:
        return False, "projected polygon is unrealistically large"
    edges = np.linalg.norm(poly - np.roll(poly, -1, axis=0), axis=1)
    if edges.min() < 10:
        return False, "projected polygon has a degenerate edge"
    return True, "ok"

def verify_geometry(q_pts, entry, matches, q_shape):
    out = {"H": None, "mask": None, "inliers": 0, "poly": None, "valid": False,
           "reason": "not enough matches for a homography"}
    if len(matches) < 4:
        return out
    src = np.float32([entry["pts"][m.trainIdx] for m in matches]).reshape(-1, 1, 2)   # reference points
    dst = np.float32([q_pts[m.queryIdx] for m in matches]).reshape(-1, 1, 2)           # query points
    H, mask = cv2.findHomography(src, dst, cv2.RANSAC, RANSAC_THRESH)
    if H is None or mask is None:
        out["reason"] = "RANSAC could not estimate a homography"
        return out
    out.update(H=H, mask=mask.ravel().astype(bool), inliers=int(mask.sum()))
    rh, rw = entry["shape"]
    corners = np.float32([[0, 0], [rw, 0], [rw, rh], [0, rh]]).reshape(-1, 1, 2)
    poly = cv2.perspectiveTransform(corners, H).reshape(-1, 2)
    ok, why = check_polygon(poly, q_shape)
    out.update(poly=poly, valid=ok, reason=why)
    return out

def recognize(img):
    """Full pipeline for one query image. Returns a dict with everything we need to report."""
    t0 = time.perf_counter()
    gray, kp, des = extract_sift(img)
    res = {"n_kp": len(kp), "kp": kp, "label": "Unknown", "decision": "REJECTED", "reasons": []}
    if des is None or len(kp) < 2:
        res["reasons"].append("no usable features in the query image")
        res["time"] = time.perf_counter() - t0
        res.update(best="-", good=0, inliers=0, ranking=[], top=None)
        return res

    q_pts = np.float32([k.pt for k in kp])
    scores = score_landmarks(des)
    ranking = sorted(scores, key=lambda n: scores[n]["good"], reverse=True)

    geo = {}
    for name in ranking[:TOP_K]:                       # geometric check for the top candidates only
        s = scores[name]
        geo[name] = verify_geometry(q_pts, s["entry"], s["matches"], img.shape)

    # best candidate = most RANSAC inliers (ties broken by match count)
    best = max(geo, key=lambda n: (geo[n]["inliers"], scores[n]["good"]))
    g, s = geo[best], scores[best]
    others = [geo[n]["inliers"] for n in geo if n != best]
    second = max(others) if others else 0

    reasons = []
    if s["good"] < MIN_GOOD:
        reasons.append(f"only {s['good']} reliable matches (need {MIN_GOOD})")
    if g["inliers"] < MIN_INLIERS:
        reasons.append(f"only {g['inliers']} RANSAC inliers (need {MIN_INLIERS})")
    if s["good"] > 0 and g["inliers"] / s["good"] < MIN_INLIER_RATIO:
        reasons.append(f"inlier ratio {g['inliers']/s['good']:.2f} is below {MIN_INLIER_RATIO}")
    if not g["valid"]:
        reasons.append("geometry unreliable: " + g["reason"])
    if g["inliers"] < MARGIN * second:
        reasons.append(f"ambiguous: runner-up has {second} inliers")

    res.update(ranking=[(n, scores[n]["good"]) for n in ranking], best=best, good=s["good"],
               inliers=g["inliers"], entry=s["entry"], matches=s["matches"], geo=g,
               top=best, reasons=reasons)
    if not reasons:
        res["decision"], res["label"] = "RECOGNIZED", best
    res["time"] = time.perf_counter() - t0
    return res

def show_result(img, res, title="", save_as=None):
    if res.get("top") is None:
        print("Rejected:", "; ".join(res["reasons"])); return
    e, g = res["entry"], res["geo"]
    ref = load_img(e["path"])
    q = img.copy()
    recognized = res["decision"] == "RECOGNIZED"
    if recognized and g["valid"]:
        cv2.polylines(q, [g["poly"].astype(np.int32).reshape(-1, 1, 2)], True, (0, 0, 255), 5)   # red polygon
    # drawMatches wants (reference, query), so swap the index order of each match
    ms = res["matches"]
    keep = list(range(len(ms)))
    if g["mask"] is not None and g["mask"].any():
        keep = [i for i in keep if g["mask"][i]]            # draw inliers only, if we have any
    keep = sorted(keep, key=lambda i: ms[i].distance)[:100]   # at most 100 lines so the figure stays readable
    dm = [cv2.DMatch(ms[i].trainIdx, ms[i].queryIdx, ms[i].distance) for i in keep]
    mask = None
    vis = cv2.drawMatches(ref, e["kp"], q, res["kp"], dm, None, matchColor=(0, 255, 0),
                          matchesMask=mask, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    plt.figure(figsize=(14, 6))
    plt.imshow(rgb(vis)); plt.axis("off")
    status = f"{res['decision']}: {res['label']}" if recognized else "REJECTED (landmark not recognized)"
    plt.title(f"{title}\n{status}   |   best candidate: {res['best']}   |   reliable matches: {res['good']}   |   "
              f"RANSAC inliers: {res['inliers']}   |   time: {res['time']:.2f} s",
              color="green" if recognized else "red", fontsize=10)
    plt.tight_layout()
    if save_as:
        plt.savefig(OUT_DIR / save_as, dpi=120)
    plt.show()
    if not recognized:
        print("Why it was rejected:")
        for r in res["reasons"]:
            print("  -", r)
        if not g["valid"]:
            print("Geometric transformation is unreliable, so no polygon was drawn.")

res = recognize(q_img)
show_result(q_img, res, title=f"Query: {demo_path.name}", save_as="task5_demo_result.png")

### Try your own photo

Put a photo in the project folder, set `MY_QUERY` to its file name and run the cell. It shows everything the assignment asks for: the query, its keypoints, the matches, the predicted landmark, the number of reliable matches, the localized region, and the recognized/rejected message.

In [ ]:
MY_QUERY = None     # example: "my_photo.jpg"

if MY_QUERY:
    img, kp, des = process_query(MY_QUERY, title="My query image")
    r = recognize(img)
    show_result(img, r, title=f"Query: {MY_QUERY}", save_as="my_query_result.png")
    print("Ranking:", r["ranking"])
else:
    print("Set MY_QUERY to a file name to test your own image.")

## Task 6: Performance evaluation

For each landmark we take a **held-out photo** (never put in the database) and create the test conditions:

| Condition | How it is made |
|---|---|
| Original | the held-out photo as it is |
| Rotated | held-out photo rotated by 45 degrees |
| Resized | held-out photo scaled to 50% |
| Low brightness | held-out photo with pixel values multiplied by 0.4 |
| Different viewpoint | a second held-out photo, taken from another place |
| Unknown landmark | photos of landmarks that are not in the database |

We record the predicted landmark, number of keypoints, reliable matches, RANSAC inliers, decision, and processing time.

In [ ]:
def rotate_bound(img, angle):
    h, w = img.shape[:2]
    c = (w / 2, h / 2)
    M = cv2.getRotationMatrix2D(c, angle, 1.0)
    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    nw, nh = int(h * sin + w * cos), int(h * cos + w * sin)
    M[0, 2] += nw / 2 - c[0]; M[1, 2] += nh / 2 - c[1]
    return cv2.warpAffine(img, M, (nw, nh), borderValue=(0, 0, 0))

def make_tests():
    tests = []
    for name in LANDMARKS:
        paths = list_images(TEST_DIR / name)
        if not paths:
            continue
        orig = load_img(paths[0])
        tests += [
            (name, "Original",             orig),
            (name, "Rotated 45 deg",       rotate_bound(orig, 45)),
            (name, "Resized 50%",          cv2.resize(orig, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)),
            (name, "Low brightness (0.4)", cv2.convertScaleAbs(orig, alpha=0.4, beta=0)),
        ]
        if len(paths) > 1:
            tests.append((name, "Different viewpoint", load_img(paths[1])))
    for p in unknown_images():
        tests.append(("Unknown", f"Not in database ({p.stem})", load_img(p)))
    return tests

tests = make_tests()
rows, results = [], []
for expected, cond, img in tests:
    r = recognize(img)
    results.append((expected, cond, img, r))
    rows.append({
        "Expected": expected, "Condition": cond,
        "Best candidate": r["best"], "Keypoints": r["n_kp"],
        "Reliable matches": r["good"], "RANSAC inliers": r["inliers"],
        "Decision": r["decision"], "Predicted": r["label"],
        "Correct": r["label"] == expected,
        "Time (s)": round(r["time"], 2),
    })
df = pd.DataFrame(rows)
df.to_csv(OUT_DIR / "results_table.csv", index=False)
df

In [ ]:
print(f"Overall accuracy: {df['Correct'].mean() * 100:.1f}%  ({df['Correct'].sum()} of {len(df)} tests)")
print(f"Average processing time: {df['Time (s)'].mean():.2f} s per query\n")

df["Group"] = df["Condition"].where(~df["Condition"].str.startswith("Not in database"), "Unknown landmark")
summary = df.groupby("Group").agg(Tests=("Correct", "size"), Correct=("Correct", "sum"),
                                  Avg_matches=("Reliable matches", "mean"),
                                  Avg_inliers=("RANSAC inliers", "mean"),
                                  Avg_time_s=("Time (s)", "mean")).round(1)
summary["Accuracy %"] = (100 * summary["Correct"] / summary["Tests"]).round(0)
display(summary)

wrong = df[~df["Correct"]][["Expected", "Condition", "Best candidate", "Reliable matches", "RANSAC inliers", "Decision"]]
print("\nFailed tests:" if len(wrong) else "\nNo failed tests.")
if len(wrong):
    display(wrong)

Result screenshots for every test (saved in `landmark_project/outputs/`). Green title means recognized, red means rejected.

In [ ]:
for i, (expected, cond, img, r) in enumerate(results):
    show_result(img, r, title=f"Expected: {expected}  |  Condition: {cond}",
                save_as=f"test_{i+1:02d}_{expected.replace(' ', '_')}_{cond.split(' ')[0].lower()}.png")

## Discussion

**What worked.** SIFT keypoints are built to survive changes in scale and rotation, and the descriptors are normalized, so they also cope with a drop in brightness. That is why the rotated, resized and darker copies of a photo usually still produce enough reliable matches and RANSAC inliers. Using geometric verification (RANSAC) on top of the ratio test is what makes the decision trustworthy: random matches do not agree on a single homography, so wrong landmarks end up with very few inliers. The unknown-landmark images should be rejected because no candidate reaches the inlier threshold or beats the runner-up by the required margin.

**Why the match count alone is not enough.** Buildings like the Colosseum or the Great Wall have repeating patterns (arches, bricks). These give many look-alike descriptors, so a wrong landmark can collect a lot of matches. Checking inliers, the inlier ratio and the margin between the top two candidates avoids this mistake.

**Limitations.**
- *Viewpoint change.* SIFT is only partly affine invariant. When the new photo is taken from a very different angle, matches drop sharply and the landmark may be rejected. A homography also assumes the surface is roughly flat, and a 3D landmark like the Statue of Liberty is not, so the drawn polygon is only an approximation.
- *Small database.* With only a few reference photos per landmark, a new viewpoint may not look like any of them. More images from many angles, distances and lighting conditions would help.
- *Speed.* Brute-force matching against every reference image grows with the database size. For a large database, use FLANN or an index structure, or reduce the descriptors.
- *Thresholds.* `MIN_GOOD`, `MIN_INLIERS` and `MARGIN` were chosen by hand. Raising them rejects more wrong images but also rejects more correct ones. They should be tuned on a larger validation set.
- *Hard conditions.* Heavy blur, night photos, strong occlusion (crowds, scaffolding), tiny landmarks in a large scene, and photos of models or paintings of the landmark can all fail.

**Practical use.** For a tourism information center this approach is cheap, needs no training, and a new landmark can be added just by dropping its photos into a folder. It is a good baseline, and the numbers in the results table above show how well it works on your own photos.

## Download everything (Google Colab)

Colab deletes its files when the session ends. This cell zips the database, screenshots and results table so you can submit them.

In [ ]:
import shutil
zip_path = shutil.make_archive("landmark_project", "zip", BASE)
print("Created", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running in Colab. The zip is in the current folder.")